In [3]:
#LIBRERIAS
%pip install ib_async

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.0
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
# --- sb_update_json_overwrite_final.py ---
from ib_async import IB, Future
from pathlib import Path
from datetime import datetime, timedelta
import pandas as pd
import json, time, re, os, tempfile

# import nest_asyncio; nest_asyncio.apply()  # ← descomenta si usas Jupyter/Spyder

# ========= CONFIG =========
HOST = "127.0.0.1"
PORT = 7496        # 7497 (paper) / 7496 (live)
CLIENT_ID = 16

# Ruta del archivo JSON (se lee y se sobrescribe aquí mismo)
JSON_PATH = Path(r"C:\Users\DanielAristizabal\Saman\Banca de Inversión - Documents\Super de Alimentos\Gestión de riesgo\Data\data_sb.json")

# Si prefieres que las claves nuevas usen 1 dígito para años 2020–2029 (ej. SBH5), deja True.
# Si quieres estándares de 2 dígitos (SBH25), pon False.
ONE_DIGIT_YEAR_2020s = True
# ==========================

# Azúcar #11 (SB) meses típicos: H=03 (Mar), K=05 (May), N=07 (Jul), V=10 (Oct)
MONTH_LETTERS = {'H':'03','K':'05','N':'07','V':'10'}
MONTH_NUM_TO_LETTER = {v:k for k,v in MONTH_LETTERS.items()}
ORDERED_MONTHS = ['H','K','N','V']

# ---------- util contrato ----------
def sb_code_to_yyyymm(code: str) -> str:
    """
    Acepta SBH5 (1 dígito, 2025) o SBH25 (2 dígitos).
    Devuelve 'YYYYMM', p.ej. '202503'.
    """
    m = re.fullmatch(r"SB([HKNV])(\d{1,2})", code.strip().upper())
    if not m:
        raise ValueError(f"Código inválido SB: {code}")
    letter, yy = m.groups()
    if len(yy) == 1:
        # Interpretación práctica para era actual: 2020-2029
        year = 2020 + int(yy)
    else:
        yy_i = int(yy)
        year = 2000 + yy_i if yy_i <= 79 else 1900 + yy_i
    mm = MONTH_LETTERS[letter]
    return f"{year}{mm}"

def yyyymm_to_sb_code(yyyymm: str) -> str:
    """
    '202503' -> 'SBH5' si ONE_DIGIT_YEAR_2020s=True y año en 2020-2029,
                 de lo contrario 'SBH25'.
    """
    y = int(yyyymm[:4]); mm = yyyymm[4:6]
    letter = MONTH_NUM_TO_LETTER[mm]
    yy2 = str(y)[-2:]
    if ONE_DIGIT_YEAR_2020s and (2020 <= y <= 2029):
        yy1 = str(y % 10)
        return f"SB{letter}{yy1}"
    return f"SB{letter}{yy2}"

def next_contract_yyyymm(curr_yyyymm: str) -> str:
    y, mm = int(curr_yyyymm[:4]), curr_yyyymm[4:6]
    letter = MONTH_NUM_TO_LETTER[mm]
    i = ORDERED_MONTHS.index(letter)
    if i == len(ORDERED_MONTHS) - 1:
        return f"{y+1}{MONTH_LETTERS['H']}"
    return f"{y}{MONTH_LETTERS[ORDERED_MONTHS[i+1]]}"

def qualify_sb(ib: IB, yyyymm: str):
    """
    Intenta calificar SB en ICEUS; si falla, prueba NYBOT (contratos muy viejos).
    """
    for ex in ("ICEUS", "NYBOT"):
        fut = Future(symbol="SB",
                     lastTradeDateOrContractMonth=yyyymm,
                     exchange=ex,
                     currency="USD",
                     includeExpired=True)
        qs = ib.qualifyContracts(fut)
        if qs:
            return qs[0]
    return None

# ---------- históricos ----------
def fetch_daily_range(ib: IB, q, start_date: str, end_dt: datetime):
    """
    Pide solo el rango faltante (diario) entre start_date y end_dt.
    Intenta TRADES, luego BID_ASK, luego MIDPOINT.
    """
    def ask(what):
        start_dt = pd.to_datetime(start_date)
        days = max(1, (end_dt - start_dt).days + 1)
        bars = ib.reqHistoricalData(
            q, endDateTime=end_dt.strftime("%Y%m%d %H:%M:%S"),
            durationStr=f"{days} D", barSizeSetting="1 day",
            whatToShow=what, useRTH=False, formatDate=2
        )
        rows = []
        for b in bars:
            d = str(b.date)
            if d < start_date:
                continue
            rows.append({
                "date": d,
                "open": b.open, "high": b.high, "low": b.low,
                "close": b.close, "volume": b.volume,
                "openinterest": None  # IB no trae OI en estas barras estándar
            })
        return rows

    for what in ("TRADES", "BID_ASK", "MIDPOINT"):
        try:
            rows = ask(what)
            if rows: return rows
        except Exception:
            pass
        time.sleep(0.25)
    return []

def fetch_full_history(ib: IB, q):
    """
    Descarga 'todo' lo práctico de una: 30Y, diario. Con fallback whatToShow.
    """
    rows = []
    for what in ("TRADES", "BID_ASK", "MIDPOINT"):
        try:
            bars = ib.reqHistoricalData(q, "", "30 Y", "1 day", what, False, 2)
            if bars:
                for b in bars:
                    rows.append({
                        "date": str(b.date), "open": b.open, "high": b.high,
                        "low": b.low, "close": b.close, "volume": b.volume,
                        "openinterest": None
                    })
                break
        except Exception:
            pass
        time.sleep(0.25)
    if not rows: return []
    df = pd.DataFrame(rows).drop_duplicates("date").sort_values("date")
    return df.to_dict(orient="records")

# ---------- JSON I/O (sobrescritura atómica) ----------
def load_db(path: Path) -> dict:
    if not path.exists():
        return {}
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def atomic_save(path: Path, data: dict):
    path.parent.mkdir(parents=True, exist_ok=True)
    fd, tmp = tempfile.mkstemp(prefix=path.name, dir=str(path.parent))
    try:
        with os.fdopen(fd, "w", encoding="utf-8") as f:
            json.dump(data, f, ensure_ascii=False, indent=2)
        os.replace(tmp, path)  # reemplazo atómico
    except Exception:
        try:
            os.remove(tmp)
        finally:
            raise

# ---------- main ----------
def main():
    db = load_db(JSON_PATH)
    if not db:
        print(f"⚠️ El archivo {JSON_PATH} está vacío o no existe.")
        return

    # Determinar último contrato en el JSON (acepta SBH5 y SBH25)
    sb_codes = []
    for c in db.keys():
        if re.fullmatch(r"SB[HKNV]\d{1,2}", c):
            try:
                _ = sb_code_to_yyyymm(c)
                sb_codes.append(c)
            except Exception:
                pass

    if not sb_codes:
        print("⚠️ No hay contratos SB válidos en el JSON.")
        return

    latest_code = max(sb_codes, key=lambda c: sb_code_to_yyyymm(c))
    latest_yyyymm = sb_code_to_yyyymm(latest_code)
    bars_latest = db.get(latest_code, [])
    last_date_latest_before = max((r["date"] for r in bars_latest), default=None)  # ✅ fecha PREVIA

    print(f"JSON: contrato SB más reciente = {latest_code}")
    print(f"Última fecha PREVIA en JSON para {latest_code}: {last_date_latest_before}")

    # Conectar IB
    ib = IB()
    try:
        ib.connect(HOST, PORT, clientId=CLIENT_ID)
    except Exception as e:
        print("No se pudo conectar a IBKR:", e)
        return

    today = datetime.now()

    # 1) Actualizar fechas faltantes del último contrato
    if latest_yyyymm and last_date_latest_before:
        q = qualify_sb(ib, latest_yyyymm)
        if q:
            start_dt = pd.to_datetime(last_date_latest_before) + timedelta(days=1)
            if start_dt.date() <= today.date():
                new_rows = fetch_daily_range(ib, q, start_dt.strftime("%Y-%m-%d"), today)
                if new_rows:
                    merged = (pd.concat([pd.DataFrame(bars_latest), pd.DataFrame(new_rows)], ignore_index=True)
                              .drop_duplicates("date").sort_values("date"))
                    db[latest_code] = merged.to_dict(orient="records")
                    print(f"{latest_code}: +{len(new_rows)} filas nuevas (hasta {merged.iloc[-1]['date']}).")
                else:
                    print(f"{latest_code}: sin filas nuevas.")
            else:
                print(f"{latest_code}: ya estaba al día.")
        else:
            print(f"No se pudo calificar {latest_code} en IB.")
    else:
        print("No se pudo determinar última fecha previa del contrato más reciente en el JSON.")

    # 2) Ver si hay contrato más nuevo; si sí, agregarlo (solo el inmediato)
    probe = next_contract_yyyymm(latest_yyyymm)
    added = False
    for _ in range(6):  # prueba hasta 6 vencimientos hacia adelante
        qnew = qualify_sb(ib, probe)
        code_new = yyyymm_to_sb_code(probe)
        if qnew:
            if code_new not in db:
                rows = fetch_full_history(ib, qnew)
                db[code_new] = rows
                print(f"Añadido {code_new}: {len(rows)} filas.")
                added = True
                break
            else:
                probe = next_contract_yyyymm(probe)  # ya existe, mira el siguiente
        else:
            probe = next_contract_yyyymm(probe)      # IB aún no lo reconoce; prueba siguiente

    # 3) Guardar (sobrescribir) el mismo archivo JSON de origen (atómico)
    atomic_save(JSON_PATH, db)
    print(f"Actualizado: {JSON_PATH}")
    if added:
        print("Se añadió un contrato nuevo.")
    else:
        print("No había contratos más nuevos que agregar.")

    # 4) Resumen
    keys = []
    for k in db.keys():
        if re.fullmatch(r"SB[HKNV]\d{1,2}", k):
            try:
                _ = sb_code_to_yyyymm(k)
                keys.append(k)
            except Exception:
                pass

    if keys:
        oldest = min(keys, key=lambda c: sb_code_to_yyyymm(c))
        newest = max(keys, key=lambda c: sb_code_to_yyyymm(c))
        last_date_newest_after = max((r["date"] for r in db[newest]), default="N/A")
        print("\nResumen:")
        print(f"- Más antiguo: {oldest}")
        print(f"- Más reciente: {newest} (última fecha del contrato tras actualizar: {last_date_newest_after})")
        print(f"- Última fecha PREVIA en JSON para {latest_code}: {last_date_latest_before}")

    if ib.isConnected():
        ib.disconnect()

if __name__ == "__main__":
    main()


JSON: contrato SB más reciente = SBN8
Última fecha PREVIA en JSON para SBN8: 2025-10-06
No se pudo conectar a IBKR: This event loop is already running


In [3]:
# MAIZ (ZC) - ACTUALIZACIÓN DE PRECIOS DESDE INTERACTIVE BROKERS
# Correcciones aplicadas (igual que en Azúcar):
# 1) NO elegir el contrato a actualizar desde el "más reciente" del JSON.
#    En su lugar, seleccionar el contrato vigente (front) por calendario y validarlo con data reciente.
# 2) endDateTime="" para pedir históricos "hasta ahora".
# 3) Reemplazo de time.sleep() por await asyncio.sleep() dentro de funciones async.
#
# En notebook: await main_async()
# En .py: asyncio.run(main_async())

from ib_async import IB, Future
from pathlib import Path
from datetime import datetime, timedelta, date
import pandas as pd
import json, re, os, tempfile
import asyncio

# ========= CONFIG =========
HOST = "127.0.0.1"
PORT = 7496        # 7497 (paper) / 7496 (live)
CLIENT_ID = 5

JSON_PATH = Path(r"C:\Users\DanielAristizabal\Saman\Banca de Inversión - Documents\Super de Alimentos\Gestión de riesgo\Data\data_zc1.json")
# ==========================

MONTH_LETTERS = {'H': '03', 'K': '05', 'N': '07', 'U': '09', 'Z': '12'}
MONTH_NUM_TO_LETTER = {v: k for k, v in MONTH_LETTERS.items()}
ORDERED_MONTHS = ['H', 'K', 'N', 'U', 'Z']
ORDERED_MM = [MONTH_LETTERS[m] for m in ORDERED_MONTHS]  # ['03','05','07','09','12']


# ---------- util contrato ----------
def code_to_yyyymm(code: str) -> str:
    m = re.fullmatch(r"ZC([HKNZU])(\d{2})", code.strip().upper())
    if not m:
        raise ValueError(f"Código inválido: {code}")
    letter, yy = m.groups()
    yy_i = int(yy)
    year = 2000 + yy_i if yy_i <= 79 else 1900 + yy_i
    mm = MONTH_LETTERS[letter]
    return f"{year}{mm}"

def yyyymm_to_code(yyyymm: str) -> str:
    y = int(yyyymm[:4])
    mm = yyyymm[4:6]
    return f"ZC{MONTH_NUM_TO_LETTER[mm]}{str(y)[-2:]}"

def next_contract_yyyymm(curr_yyyymm: str) -> str:
    y, mm = int(curr_yyyymm[:4]), curr_yyyymm[4:6]
    letter = MONTH_NUM_TO_LETTER[mm]
    i = ORDERED_MONTHS.index(letter)
    if i == len(ORDERED_MONTHS) - 1:
        return f"{y+1}{MONTH_LETTERS['H']}"
    return f"{y}{MONTH_LETTERS[ORDERED_MONTHS[i+1]]}"

async def qualify_zc_async(ib: IB, yyyymm: str):
    # Para maíz usualmente CBOT es correcto. Dejamos ECBOT como fallback.
    for ex in ("CBOT", "ECBOT"):
        fut = Future(
            symbol="ZC",
            lastTradeDateOrContractMonth=yyyymm,
            exchange=ex,
            currency="USD",
            includeExpired=True
        )
        try:
            qs = await ib.qualifyContractsAsync(fut)
            if qs:
                return qs[0]
        except Exception:
            pass
        await asyncio.sleep(0.05)
    return None


# ---------- selección de contrato vigente (front) por calendario ----------
def calendar_candidates_from_today(n: int = 12) -> list[str]:
    """
    Genera yyyymm candidatos de ZC (H/K/N/U/Z) a partir de hoy.
    """
    today = date.today()
    y = today.year
    m = today.month

    def first_for_year(year: int, start_month: int) -> str | None:
        for mm in ORDERED_MM:
            if int(mm) >= start_month:
                return f"{year}{mm}"
        return None

    first = first_for_year(y, m)
    if first is None:
        first = f"{y+1}03"

    out = [first]
    while len(out) < n:
        out.append(next_contract_yyyymm(out[-1]))
    return out

async def has_recent_data(ib: IB, q, lookback_days: int = 25, tolerance_days: int = 15) -> bool:
    """
    Verifica que el contrato tenga barra diaria reciente.
    """
    try:
        bars = await ib.reqHistoricalDataAsync(
            q,
            endDateTime="",
            durationStr=f"{lookback_days} D",
            barSizeSetting="1 day",
            whatToShow="TRADES",
            useRTH=False,
            formatDate=2
        )
        if not bars:
            return False
        last_bar_date = pd.to_datetime(str(bars[-1].date)).date()
        return last_bar_date >= (date.today() - timedelta(days=tolerance_days))
    except Exception:
        return False

async def pick_front_contract_async(ib: IB) -> tuple[str, object]:
    """
    Elige el primer yyyymm candidato que:
    - califique en IBKR y
    - tenga data reciente (evita contratos muy lejanos o sin data).
    """
    candidates = calendar_candidates_from_today(n=14)
    for yyyymm in candidates:
        q = await qualify_zc_async(ib, yyyymm)
        if not q:
            continue
        if await has_recent_data(ib, q):
            return yyyymm, q
    return "", None


# ---------- históricos ----------
async def fetch_daily_range_async(ib: IB, q, start_date: str, end_dt: datetime):
    async def ask(what):
        start_dt = pd.to_datetime(start_date)
        days = max(1, (end_dt - start_dt).days + 1)

        bars = await ib.reqHistoricalDataAsync(
            q,
            endDateTime="",  # hasta ahora
            durationStr=f"{days} D",
            barSizeSetting="1 day",
            whatToShow=what,
            useRTH=False,
            formatDate=2
        )

        rows = []
        for b in bars:
            d = str(b.date)
            if d < start_date:
                continue
            rows.append({
                "date": d,
                "open": b.open, "high": b.high, "low": b.low,
                "close": b.close, "volume": b.volume,
                "openinterest": None
            })
        return rows

    for what in ("TRADES", "BID_ASK", "MIDPOINT"):
        try:
            rows = await ask(what)
            if rows:
                return rows
        except Exception:
            pass
        await asyncio.sleep(0.25)
    return []

async def fetch_full_history_async(ib: IB, q):
    rows = []
    for what in ("TRADES", "BID_ASK", "MIDPOINT"):
        try:
            bars = await ib.reqHistoricalDataAsync(
                q,
                endDateTime="",
                durationStr="30 Y",
                barSizeSetting="1 day",
                whatToShow=what,
                useRTH=False,
                formatDate=2
            )
            if bars:
                for b in bars:
                    rows.append({
                        "date": str(b.date),
                        "open": b.open, "high": b.high,
                        "low": b.low, "close": b.close,
                        "volume": b.volume,
                        "openinterest": None
                    })
                break
        except Exception:
            pass
        await asyncio.sleep(0.25)

    if not rows:
        return []
    df = pd.DataFrame(rows).drop_duplicates("date").sort_values("date")
    return df.to_dict(orient="records")


# ---------- JSON I/O ----------
def load_db(path: Path) -> dict:
    if not path.exists():
        return {}
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def atomic_save(path: Path, data: dict):
    path.parent.mkdir(parents=True, exist_ok=True)
    fd, tmp = tempfile.mkstemp(prefix=path.name, dir=str(path.parent))
    try:
        with os.fdopen(fd, "w", encoding="utf-8") as f:
            json.dump(data, f, ensure_ascii=False, indent=2)
        os.replace(tmp, path)
    except Exception:
        try:
            os.remove(tmp)
        finally:
            raise


# ---------- main async ----------
async def main_async():
    db = load_db(JSON_PATH)
    if not db:
        print(f"⚠️ El archivo {JSON_PATH} está vacío o no existe.")
        return

    ib = IB()
    try:
        await ib.connectAsync(HOST, PORT, clientId=CLIENT_ID)
    except Exception as e:
        print("No se pudo conectar a IBKR:", e)
        return

    try:
        # 1) Escoger contrato front vigente (por calendario + data reciente)
        front_yyyymm, qfront = await pick_front_contract_async(ib)
        if not qfront:
            print("⚠️ No pude encontrar un contrato ZC vigente con data reciente.")
            print("   Posibles causas: permisos de datos CBOT, símbolo/exchange, o data no disponible.")
            return

        front_code = yyyymm_to_code(front_yyyymm)
        print(f"Front seleccionado: {front_code} ({front_yyyymm}) | conId={qfront.conId} | exch={qfront.exchange}")

        # 2) Actualizar/crear el front en el JSON
        today_dt = datetime.now()
        bars_front = db.get(front_code, [])
        last_date_front_before = max((r["date"] for r in bars_front), default=None)
        print(f"Última fecha PREVIA en JSON para {front_code}: {last_date_front_before}")

        if last_date_front_before:
            start_dt = pd.to_datetime(last_date_front_before) + timedelta(days=1)
            if start_dt.date() <= today_dt.date():
                new_rows = await fetch_daily_range_async(ib, qfront, start_dt.strftime("%Y-%m-%d"), today_dt)
                if new_rows:
                    merged = (pd.concat([pd.DataFrame(bars_front), pd.DataFrame(new_rows)], ignore_index=True)
                              .drop_duplicates("date")
                              .sort_values("date"))
                    db[front_code] = merged.to_dict(orient="records")
                    print(f"{front_code}: +{len(new_rows)} filas nuevas (hasta {merged.iloc[-1]['date']}).")
                else:
                    print(f"{front_code}: sin filas nuevas (posible: aún no hay barra diaria de hoy).")
            else:
                print(f"{front_code}: ya estaba al día.")
        else:
            rows = await fetch_full_history_async(ib, qfront)
            db[front_code] = rows
            last_after = max((r["date"] for r in rows), default="N/A")
            print(f"{front_code}: creado en JSON con {len(rows)} filas (última fecha: {last_after}).")

        # 3) (Opcional) Agregar 2 contratos hacia adelante
        probe = next_contract_yyyymm(front_yyyymm)
        added = 0
        for _ in range(10):
            code_new = yyyymm_to_code(probe)
            if code_new in db:
                probe = next_contract_yyyymm(probe)
                continue

            qnew = await qualify_zc_async(ib, probe)
            if qnew:
                rows = await fetch_full_history_async(ib, qnew)
                db[code_new] = rows
                print(f"Añadido {code_new}: {len(rows)} filas.")
                added += 1
                if added >= 2:
                    break

            probe = next_contract_yyyymm(probe)

        atomic_save(JSON_PATH, db)
        print(f"Actualizado: {JSON_PATH}")
        print(f"Contratos nuevos añadidos: {added}")

        # 4) Resumen (solo informativo)
        keys = [k for k in db.keys() if re.fullmatch(r"ZC[HKNZU]\d{2}", k)]
        if keys:
            newest = max(keys, key=lambda c: code_to_yyyymm(c))
            last_date_newest_after = max((r["date"] for r in db.get(newest, [])), default="N/A")
            print("\nResumen:")
            print(f"- Más reciente (por código) en JSON: {newest} (última fecha: {last_date_newest_after})")
            print(f"- Front actualizado: {front_code}")

    finally:
        if ib.isConnected():
            ib.disconnect()


# En notebook:
await main_async()


Front seleccionado: ZCH26 (202603) | conId=671574012 | exch=CBOT
Última fecha PREVIA en JSON para ZCH26: 2026-03-03
ZCH26: ya estaba al día.
Actualizado: C:\Users\DanielAristizabal\Saman\Banca de Inversión - Documents\Super de Alimentos\Gestión de riesgo\Data\data_zc1.json
Contratos nuevos añadidos: 0

Resumen:
- Más reciente (por código) en JSON: ZCZ29 (última fecha: 2026-03-02)
- Front actualizado: ZCH26


In [1]:
# MAIZ (ZC) - ACTUALIZACIÓN DE PRECIOS DESDE INTERACTIVE BROKERS
from ib_async import IB, Future
from pathlib import Path
from datetime import datetime, timedelta, date
import pandas as pd
import json, re, os, tempfile
import asyncio

# ========= CONFIG =========
HOST = "127.0.0.1"
PORT = 7496        # 7497 (paper) / 7496 (live)
CLIENT_ID = 5

JSON_PATH = Path(r"C:\Users\DanielAristizabal\Saman\Banca de Inversión - Documents\Super de Alimentos\Gestión de riesgo\Data\data_zc1.json")

# FECHA MÍNIMA - Solo traer datos desde esta fecha
START_DATE = "2025-01-01"
# ==========================

MONTH_LETTERS = {'H': '03', 'K': '05', 'N': '07', 'U': '09', 'Z': '12'}
MONTH_NUM_TO_LETTER = {v: k for k, v in MONTH_LETTERS.items()}
ORDERED_MONTHS = ['H', 'K', 'N', 'U', 'Z']

# ---------- util contrato ----------
def code_to_yyyymm(code: str) -> str:
    m = re.fullmatch(r"ZC([HKNUZ])(\d{2})", code.strip().upper())
    if not m:
        raise ValueError(f"Código inválido: {code}")
    letter, yy = m.groups()
    yy_i = int(yy)
    year = 2000 + yy_i if yy_i <= 79 else 1900 + yy_i
    return f"{year}{MONTH_LETTERS[letter]}"

def yyyymm_to_code(yyyymm: str) -> str:
    y = int(yyyymm[:4])
    mm = yyyymm[4:6]
    return f"ZC{MONTH_NUM_TO_LETTER[mm]}{str(y)[-2:]}"

async def qualify_zc_async(ib: IB, yyyymm: str):
    for ex in ("CBOT", "ECBOT"):
        fut = Future(
            symbol="ZC",
            lastTradeDateOrContractMonth=yyyymm,
            exchange=ex,
            currency="USD",
            includeExpired=True
        )
        try:
            qs = await ib.qualifyContractsAsync(fut)
            if qs:
                return qs[0]
        except Exception:
            pass
        await asyncio.sleep(0.05)
    return None

# ---------- históricos ----------
async def fetch_history_from_date(ib: IB, q, start_date: str):
    """Descarga histórico desde start_date hasta hoy"""
    start_dt = pd.to_datetime(start_date)
    end_dt = datetime.now()
    days = max(1, (end_dt - start_dt).days + 1)
    
    for what in ("TRADES", "BID_ASK", "MIDPOINT"):
        try:
            bars = await ib.reqHistoricalDataAsync(
                q, endDateTime="", durationStr=f"{days} D",
                barSizeSetting="1 day", whatToShow=what,
                useRTH=False, formatDate=2
            )
            if bars:
                rows = []
                for b in bars:
                    d = str(b.date)
                    if d >= start_date:  # Solo fechas >= START_DATE
                        rows.append({
                            "date": d, "open": b.open, "high": b.high, "low": b.low,
                            "close": b.close, "volume": b.volume, "openinterest": None
                        })
                if rows:
                    df = pd.DataFrame(rows).drop_duplicates("date").sort_values("date")
                    return df.to_dict(orient="records")
        except Exception:
            pass
        await asyncio.sleep(0.25)
    return []

# ---------- JSON I/O ----------
def load_db(path: Path) -> dict:
    if not path.exists():
        return {}
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def atomic_save(path: Path, data: dict):
    path.parent.mkdir(parents=True, exist_ok=True)
    fd, tmp = tempfile.mkstemp(prefix=path.name, dir=str(path.parent))
    try:
        with os.fdopen(fd, "w", encoding="utf-8") as f:
            json.dump(data, f, ensure_ascii=False, indent=2)
        os.replace(tmp, path)
    except Exception:
        try:
            os.remove(tmp)
        finally:
            raise

# ---------- main async ----------
async def main_async():
    db = load_db(JSON_PATH)
    
    ib = IB()
    try:
        await ib.connectAsync(HOST, PORT, clientId=CLIENT_ID)
    except Exception as e:
        print("No se pudo conectar a IBKR:", e)
        return

    try:
        today = datetime.now()
        
        # Contratos existentes en JSON
        zc_contracts = [k for k in db.keys() if re.fullmatch(r"ZC[HKNUZ]\d{2}", k)]
        print(f"Contratos en JSON: {len(zc_contracts)}")
        print(f"Fecha mínima: {START_DATE}\n")
        
        updated = 0
        
        for code in sorted(zc_contracts):
            try:
                yyyymm = code_to_yyyymm(code)
            except ValueError:
                continue
            
            q = await qualify_zc_async(ib, yyyymm)
            if not q:
                print(f"⚠️ {code}: no calificado en IBKR")
                continue
            
            # Obtener última fecha en JSON
            bars = db.get(code, [])
            # Filtrar datos anteriores a START_DATE
            bars = [r for r in bars if r["date"] >= START_DATE]
            last_date = max((r["date"] for r in bars), default=None)
            
            if last_date:
                # Ya tiene datos, buscar desde última fecha + 1
                start = (pd.to_datetime(last_date) + timedelta(days=1)).strftime("%Y-%m-%d")
                if start <= today.strftime("%Y-%m-%d"):
                    new_rows = await fetch_history_from_date(ib, q, start)
                    if new_rows:
                        merged = (pd.concat([pd.DataFrame(bars), pd.DataFrame(new_rows)], ignore_index=True)
                                  .drop_duplicates("date").sort_values("date"))
                        db[code] = merged.to_dict(orient="records")
                        print(f"✅ {code}: +{len(new_rows)} filas (hasta {merged.iloc[-1]['date']})")
                        updated += 1
                    else:
                        print(f"   {code}: al día ({last_date})")
                else:
                    print(f"   {code}: al día ({last_date})")
            else:
                # Sin datos, descargar desde START_DATE
                rows = await fetch_history_from_date(ib, q, START_DATE)
                if rows:
                    db[code] = rows
                    print(f"✅ {code}: {len(rows)} filas (desde {START_DATE})")
                    updated += 1
                else:
                    print(f"⚠️ {code}: sin datos")
            
            await asyncio.sleep(0.5)
        
        atomic_save(JSON_PATH, db)
        
        print(f"\n{'='*50}")
        print(f"Actualizados: {updated} | Archivo: {JSON_PATH.name}")
        print(f"{'='*50}")
        
        print("\nÚltima fecha por contrato:")
        for code in sorted(zc_contracts, key=lambda x: code_to_yyyymm(x)):
            bars = db.get(code, [])
            last = max((r["date"] for r in bars), default="N/A")
            print(f"  {code}: {last}")

    finally:
        if ib.isConnected():
            ib.disconnect()

await main_async()

Contratos en JSON: 20
Fecha mínima: 2025-01-01

   ZCH25: al día (2025-03-14)


C:\Users\DanielAristizabal\AppData\Local\Temp\ipykernel_26296\3487146994.py:152: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  merged = (pd.concat([pd.DataFrame(bars), pd.DataFrame(new_rows)], ignore_index=True)


✅ ZCH26: +10 filas (hasta 2026-03-03)


C:\Users\DanielAristizabal\AppData\Local\Temp\ipykernel_26296\3487146994.py:152: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  merged = (pd.concat([pd.DataFrame(bars), pd.DataFrame(new_rows)], ignore_index=True)


✅ ZCH27: +10 filas (hasta 2026-03-03)
✅ ZCH28: +10 filas (hasta 2026-03-03)
   ZCK25: al día (2025-05-14)


...

✅ ZCZ26: +10 filas (hasta 2026-03-03)


C:\Users\DanielAristizabal\AppData\Local\Temp\ipykernel_26296\3487146994.py:152: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  merged = (pd.concat([pd.DataFrame(bars), pd.DataFrame(new_rows)], ignore_index=True)


✅ ZCZ27: +10 filas (hasta 2026-03-03)


C:\Users\DanielAristizabal\AppData\Local\Temp\ipykernel_26296\3487146994.py:152: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  merged = (pd.concat([pd.DataFrame(bars), pd.DataFrame(new_rows)], ignore_index=True)


✅ ZCZ28: +10 filas (hasta 2026-03-03)
✅ ZCZ29: +10 filas (hasta 2026-03-02)

Actualizados: 15 | Archivo: data_zc1.json

Última fecha por contrato:
  ZCH25: 2025-03-14
  ZCK25: 2025-05-14
  ZCN25: 2025-07-15
  ZCU25: 2025-09-12
  ZCZ25: 2025-12-12
  ZCH26: 2026-03-03
  ZCK26: 2026-03-03
  ZCN26: 2026-03-03
  ZCU26: 2026-03-03
  ZCZ26: 2026-03-03
  ZCH27: 2026-03-03
  ZCK27: 2026-03-03
  ZCN27: 2026-03-03
  ZCU27: 2026-03-03
  ZCZ27: 2026-03-03
  ZCH28: 2026-03-03
  ZCN28: 2026-03-02
  ZCZ28: 2026-03-03
  ZCN29: 2026-03-02
  ZCZ29: 2026-03-02


In [2]:
# PRECIOS DEL ÚLTIMO DÍA DEL MES ANTERIOR - TODOS LOS FUTUROS DE MAÍZ
from datetime import date
from dateutil.relativedelta import relativedelta
import pandas as pd
import json
from pathlib import Path
import re

JSON_PATH = Path(r"C:\Users\DanielAristizabal\Saman\Banca de Inversión - Documents\Super de Alimentos\Gestión de riesgo\Data\data_zc1.json")

# Cargar datos
with open(JSON_PATH, "r", encoding="utf-8") as f:
    db = json.load(f)

# Calcular mes anterior
hoy = date.today()
primer_dia_mes_actual = hoy.replace(day=1)
ultimo_dia_mes_anterior = primer_dia_mes_actual - relativedelta(days=1)
mes_anterior = ultimo_dia_mes_anterior.strftime("%Y-%m")

print(f"Buscando precios del mes: {mes_anterior}")
print(f"(Último día del mes anterior: {ultimo_dia_mes_anterior})\n")

# Recopilar precios del último día del mes anterior para cada contrato
resultados = []

zc_contracts = [k for k in db.keys() if re.fullmatch(r"ZC[HKNUZ]\d{2}", k)]

for code in sorted(zc_contracts):
    bars = db.get(code, [])
    
    # Filtrar solo fechas del mes anterior
    fechas_mes = [r for r in bars if r["date"].startswith(mes_anterior)]
    
    if fechas_mes:
        # Obtener el último día disponible del mes
        ultimo_registro = max(fechas_mes, key=lambda x: x["date"])
        resultados.append({
            "Contrato": code,
            "Fecha": ultimo_registro["date"],
            "Close": ultimo_registro["close"],
            "Volume": ultimo_registro["volume"]
        })
    else:
        resultados.append({
            "Contrato": code,
            "Fecha": "Sin datos",
            "Close": None,
            "Volume": None
        })

# Crear DataFrame
df_ultimo_dia_mes = pd.DataFrame(resultados)
print(f"{'='*60}")
print(f"PRECIOS ÚLTIMO DÍA DEL MES ANTERIOR ({mes_anterior})")
print(f"{'='*60}")
display(df_ultimo_dia_mes)

Buscando precios del mes: 2026-02
(Último día del mes anterior: 2026-02-28)

PRECIOS ÚLTIMO DÍA DEL MES ANTERIOR (2026-02)


,Contrato,Fecha,Close,Volume
0,ZCH25,Sin datos,NaN,NaN
1,ZCH26,2026-02-27,438.75,3848.0
2,ZCH27,2026-02-27,480.75,6915.0
3,ZCH28,2026-02-27,488.50,7.0
4,ZCK25,Sin datos,NaN,NaN
5,ZCK26,2026-02-27,448.50,181339.0
6,ZCK27,2026-02-27,487.00,418.0
7,ZCN25,Sin datos,NaN,NaN
8,ZCN26,2026-02-27,456.00,51040.0
9,ZCN27,2026-02-27,490.00,275.0
